# ragas 테스트셋 검증하기

ragas 가 만든 질문은 **LLM 이 만든 초안**입니다. 평가에 쓰기 전에 아래 순서로 걸러냅니다.

| 단계 | 무엇을 확인하나 | 방법 | 비용 |
|---|---|---|---|
| 1 | 한국어 질문인가 | 한글 비율 계산 | 무료 |
| 2 | 비슷한 질문이 중복되지 않나 | 질문 임베딩 유사도 | 매우 저렴 |
| 3 | 원문을 그대로 베낀 쉬운 질문은 아닌가 | 질문과 근거 문서의 글자 겹침 | 무료 |
| 4 | **정답(reference)이 근거 문서에 충실한가** | Faithfulness 를 “정답 vs 근거 문서”에 적용 | LLM 호출 |
| 5 | 사람 검수 | 검수용 CSV 로 내보내 직접 체크 | 사람 시간 |
| 6 | “모르는 걸 모른다고 하나” 문항 추가 | 문서에 없는 질문 직접 작성 | 무료 |
| 7 | 최종 테스트셋 저장 | 통과 문항 + 추가 문항 | - |

> 🧭 **따라 하는 법**: 셀을 위에서부터 하나씩 실행하고, 각 단계의 **“✋ 직접 해 보기”** 에서 기준값을 바꿔 결과가 어떻게 달라지는지 확인해 보세요.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## 0. 테스트셋 불러오기

두 파일을 합쳐서 검증합니다.
- `ragas_synthetic_dataset.csv`: ch05_01 에서 만든 10문항 (한국어/영어가 섞여 있음 → 검증 연습용으로 좋음)
- `ragas_testset_ko.csv`: ch05_03 에서 한국어로 만든 4문항

In [ ]:
import ast

import pandas as pd

frames = []
for path in ["../data/ragas_synthetic_dataset.csv", "../data/ragas_testset_ko.csv"]:
    d = pd.read_csv(path)
    d["source_file"] = path.split("/")[-1]  # 어느 파일에서 왔는지 기록
    frames.append(d)

df = pd.concat(frames, ignore_index=True)

# CSV 로 저장하면 리스트가 문자열이 되므로 다시 리스트로 복원
df["reference_contexts"] = df["reference_contexts"].apply(
    lambda v: ast.literal_eval(v) if isinstance(v, str) else v
)

print(f"총 {len(df)}문항")
df[["user_input", "synthesizer_name", "source_file"]]

검증 결과를 한 곳에 모으기 위해 **문항마다 문제 목록(`issues`)** 을 만들어 두고, 단계마다 문제가 발견되면 여기에 추가합니다.

In [ ]:
df["issues"] = [[] for _ in range(len(df))]


def flag(mask, reason):
    """mask 가 True 인 문항에 문제 사유를 추가합니다."""
    for i in df.index[mask]:
        df.at[i, "issues"].append(reason)
    print(f"[{reason}] {int(mask.sum())}문항")

---
## 1단계. 언어 검사

한국어 문서로 만든 테스트셋인데 영어 질문이 섞이면, 실제 사용자 질문과 달라서 평가가 왜곡됩니다.
질문에서 **한글이 차지하는 비율**로 판단합니다.

In [ ]:
import re


def korean_ratio(text: str) -> float:
    """공백을 뺀 글자 중 한글 비율"""
    chars = re.sub(r"\s", "", text)
    return len(re.findall(r"[가-힣]", chars)) / max(len(chars), 1)


KO_MIN = 0.5  # 한글 비율 기준

df["ko_ratio"] = df["user_input"].apply(korean_ratio).round(2)
flag(df["ko_ratio"] < KO_MIN, "영어 질문")

df[["user_input", "ko_ratio"]].sort_values("ko_ratio")

> ✋ **직접 해 보기**: `KO_MIN` 을 `0.3` 으로 낮춰 보세요. “What role did 매사추세츠 공과대 (MIT)…” 처럼 **한영이 섞인 질문**이 통과하는지 확인하고, 우리 서비스에서 허용할지 정해 보세요.

---
## 2단계. 중복 질문 검사

거의 같은 질문이 여러 개 있으면 그 주제의 점수가 **여러 번 반영**되어 평균이 치우칩니다.
질문을 임베딩하고, **코사인 유사도가 기준 이상인 쌍**을 찾습니다.

In [ ]:
import numpy as np
from langchain_openai import OpenAIEmbeddings

emb = OpenAIEmbeddings(model="text-embedding-3-small")
vecs = np.array(emb.embed_documents(df["user_input"].tolist()))

# 정규화된 벡터끼리 내적 = 코사인 유사도
vecs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)
sim = vecs @ vecs.T

DUP_MIN = 0.7  # 이 이상이면 중복으로 봄 (text-embedding-3-small 은 유사도가 낮게 나오는 편)

pairs = []
for i in range(len(df)):
    for j in range(i + 1, len(df)):
        pairs.append((round(float(sim[i, j]), 3), i, j))
pairs.sort(reverse=True)

# 유사도가 가장 높은 쌍 5개 확인
for s, i, j in pairs[:5]:
    mark = "⚠️ 중복" if s >= DUP_MIN else ""
    print(f"{s:.3f} {mark}\n  [{i}] {df.at[i, 'user_input']}\n  [{j}] {df.at[j, 'user_input']}\n")

In [ ]:
# 중복 쌍에서는 뒤쪽 문항만 표시 (앞쪽 하나는 남김)
dup_idx = {j for s, i, j in pairs if s >= DUP_MIN}
flag(df.index.isin(dup_idx), "중복 질문")

> ✋ **직접 해 보기**: 위에 출력된 상위 5쌍을 읽고, **사람이 보기에 중복인 쌍이 어디까지인지** 판단해서 `DUP_MIN` 을 조정해 보세요.
> 임베딩 유사도는 “같은 주제”와 “같은 질문”을 잘 구분하지 못하므로 기준은 데이터를 보고 정해야 합니다.
>
> 실제 예: “바이든 대통령이 **발표한** AI 행정명령의 주요 내용은?”과 “… 바이든 대통령이 **서명한** 행정명령의 주요 내용은?”은
> 사실상 같은 질문인데 유사도가 약 **0.75** 였습니다. 흔히 쓰는 0.85~0.9 기준이면 놓칩니다.

---
## 3단계. 베끼기 검사 (너무 쉬운 질문)

질문이 근거 문서 문장을 거의 그대로 가져왔다면, 검색기는 **단어만 맞춰도** 쉽게 찾습니다.
이런 문항이 많으면 검색 점수가 실제보다 높게 나옵니다.

질문의 **2글자 조각(bigram)** 중 몇 %가 근거 문서에도 있는지로 판단합니다.

In [ ]:
def bigrams(text: str) -> set:
    t = re.sub(r"\s", "", text)
    return {t[i : i + 2] for i in range(len(t) - 1)}


def copy_ratio(question: str, contexts: list) -> float:
    """질문의 2글자 조각 중 근거 문서에도 나오는 비율 (1.0 = 전부 원문에 있음)"""
    q = bigrams(question)
    ctx = bigrams(" ".join(contexts))
    return len(q & ctx) / max(len(q), 1)


COPY_MAX = 0.85  # 이보다 높으면 "베낀 질문"으로 봄

df["copy_ratio"] = [round(copy_ratio(q, c), 2) for q, c in zip(df["user_input"], df["reference_contexts"])]
flag(df["copy_ratio"] > COPY_MAX, "원문 베끼기")

df[["user_input", "copy_ratio"]].sort_values("copy_ratio", ascending=False)

> 💡 오늘 데이터에서는 가장 높은 값이 약 0.65 라 걸린 문항이 없었습니다. 정상입니다. ragas 는 질문을 원문과 다르게 쓰도록 되어 있습니다.
>
> ✋ **직접 해 보기**: 영어 질문은 한국어 원문과 글자가 겹치지 않아 `copy_ratio` 가 낮게 나옵니다. **1단계를 먼저 통과한 문항**에서만 이 값을 의미 있게 볼 수 있다는 점을 확인해 보세요.

---
## 4단계. 정답(reference) 검증 ⭐ 가장 중요

ragas 가 만든 **정답이 틀리면**, 우리 RAG 가 맞게 답해도 틀렸다고 채점됩니다.

**아이디어**: `Faithfulness`(답변이 문서에 근거했나)를 **정답에** 적용합니다.
- `response` 자리에 → `reference`(ragas 가 만든 정답)
- `retrieved_contexts` 자리에 → `reference_contexts`(ragas 가 정답을 만들 때 본 문서)

점수가 낮으면 **정답에 문서에 없는 내용이 섞였다**는 뜻입니다.

> 💡 생성에 쓴 모델(gpt-4o-mini)과 **다른 모델**로 채점하면 “같은 모델이 문제도 내고 채점도 하는” 순환 문제를 줄일 수 있습니다. 여기서는 gpt-4o 를 사용합니다.

In [ ]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import Faithfulness

import asyncio

judge_llm = llm_factory("gpt-4o", client=AsyncOpenAI())
faithfulness = Faithfulness(llm=judge_llm)

# ⚠️ abatch_score 로 한꺼번에 보내면 gpt-4o 분당 토큰 한도(TPM)를 넘어 429 에러가 나기 쉽습니다.
#    동시에 2건만 보내고, 429 가 나면 잠깐 기다렸다 다시 시도합니다.
#    (실행 중 "API call failed ... 429" 로그가 보여도, 마지막에 "채점 실패" 가 없으면 재시도로 모두 성공한 것입니다.)
sem = asyncio.Semaphore(2)


async def score_one(q, ref, ctx, retries=5):
    async with sem:
        for attempt in range(retries):
            try:
                r = await faithfulness.ascore(user_input=q, response=ref, retrieved_contexts=ctx)
                return round(r.value, 2) if r.value is not None else None
            except Exception as e:
                if "429" in str(e) and attempt < retries - 1:
                    await asyncio.sleep(10 * (attempt + 1))  # 10초, 20초 … 기다린 뒤 재시도
                    continue
                print("채점 실패:", q[:40], "→", str(e)[:80])
                return None


df["ref_faithfulness"] = await asyncio.gather(*[
    score_one(q, ref, ctx)
    for q, ref, ctx in zip(df["user_input"], df["reference"], df["reference_contexts"])
])

In [ ]:
REF_MIN = 0.8  # 정답의 80% 이상이 근거 문서로 확인돼야 통과

flag(df["ref_faithfulness"].fillna(0) < REF_MIN, "정답이 근거에 없음")
df[["user_input", "ref_faithfulness"]].sort_values("ref_faithfulness")

In [ ]:
# 점수가 가장 낮은 문항의 정답과 근거 문서를 직접 비교해 봅니다.
worst = df["ref_faithfulness"].idxmin()
print("질문:", df.at[worst, "user_input"])
print("\n정답(reference):\n", df.at[worst, "reference"])
print("\n근거 문서(앞 600자):\n", " ".join(df.at[worst, "reference_contexts"])[:600])

> ✋ **직접 해 보기**: 위 문항의 정답에서 **근거 문서에 없는 문장**을 직접 찾아보세요. LLM 채점이 맞았는지 사람이 확인하는 것이 5단계의 연습이 됩니다.

---
## 자동 검증 결과 요약

지금까지 발견된 문제를 한 표로 봅니다.

In [ ]:
df["auto_pass"] = df["issues"].apply(len) == 0

summary = df[["user_input", "ko_ratio", "copy_ratio", "ref_faithfulness", "issues", "auto_pass"]]
print(f"자동 검증 통과: {int(df['auto_pass'].sum())} / {len(df)}문항")
summary

In [ ]:
# 어떤 문제가 몇 번 나왔나
from collections import Counter

Counter(reason for issues in df["issues"] for reason in issues)

---
## 5단계. 사람 검수

자동 검증은 **명백한 문제만** 걸러냅니다. 질문이 자연스러운지, 정답이 충분한지는 사람이 봐야 합니다.

아래 셀은 **자동 검증을 통과한 문항**을 검수용 CSV 로 내보냅니다.
엑셀이나 VSCode 에서 열고, 체크 칸에 `O` / `X` 를 채워 저장하세요.

| 칸 | 기준 |
|---|---|
| `answerable` | 근거 문서만 읽고 답할 수 있다 |
| `correct` | 정답이 문서와 일치하고 빠진 내용이 없다 |
| `natural` | 실제 사용자가 할 법한 자연스러운 질문이다 |
| `not_trivial` | 원문 문장을 그대로 옮긴 질문이 아니다 |
| `note` | 고칠 점 메모 (정답을 직접 고쳐 써도 됨) |

In [ ]:
review = df[df["auto_pass"]][["user_input", "reference", "reference_contexts", "synthesizer_name"]].copy()
for col in ["answerable", "correct", "natural", "not_trivial", "note"]:
    review[col] = ""

REVIEW_PATH = "../data/testset_review.csv"
review.to_csv(REVIEW_PATH, index=False, encoding="utf-8-sig")  # utf-8-sig: 엑셀에서 한글 안 깨짐
print(f"{len(review)}문항을 {REVIEW_PATH} 로 내보냈습니다. 검수 후 다음 셀을 실행하세요.")

In [ ]:
# 검수한 CSV 를 다시 읽어서, 4개 칸이 모두 O 인 문항만 통과시킵니다.
reviewed = pd.read_csv(REVIEW_PATH)
checks = ["answerable", "correct", "natural", "not_trivial"]

if reviewed[checks].isna().all().all():
    print("아직 검수 전입니다. 지금은 자동 검증 통과 문항을 그대로 사용합니다.")
    human_pass = reviewed
else:
    ok = (reviewed[checks].astype(str).apply(lambda s: s.str.strip().str.upper()) == "O").all(axis=1)
    human_pass = reviewed[ok]
    fail_rate = 1 - ok.mean()
    print(f"사람 검수 통과: {int(ok.sum())} / {len(reviewed)} (불합격률 {fail_rate:.0%})")
    if fail_rate > 0.2:
        print("⚠️ 불합격률이 20%를 넘습니다. 생성 설정(페르소나, llm_context, 모델)을 바꿔 다시 생성하는 것을 권장합니다.")

human_pass[["user_input"]]

---
## 6단계. “문서에 없는 질문” 추가

ragas 는 **문서로 답할 수 있는 질문만** 만듭니다. 그래서 RAG 가 **모르는 것을 모른다고 하는지**(환각을 하지 않는지)는 확인할 수 없습니다.
이런 문항은 직접 몇 개 추가합니다.

> 채점 방법: 이 문항들은 `reference_contexts` 가 비어 있으므로 ContextRecall 대신,
> **답변이 “확인할 수 없다”는 취지인지** 를 `FactualCorrectness` 나 사람이 확인합니다.

In [ ]:
NOT_IN_DOC = "제공된 문서에서는 확인할 수 없습니다."

negatives = pd.DataFrame({
    "user_input": [
        "SPRi AI Brief 12월호에 소개된 애플의 생성 AI 모델 이름은 무엇인가요?",  # 문서에 애플 모델 없음
        "2024년 EU AI Act 최종 시행일은 언제인가요?",                            # 문서 발행 이후 사건
        "삼성 가우스의 월 구독 요금은 얼마인가요?",                              # 문서에 가격 정보 없음
    ],
    "reference": [NOT_IN_DOC] * 3,
    "reference_contexts": [[] for _ in range(3)],
    "synthesizer_name": ["manual_negative"] * 3,
})
negatives

> ✋ **직접 해 보기**: 문서를 훑어보고 **그럴듯하지만 문서에 없는 질문**을 2개 더 추가해 보세요. 문서에 있는 회사·기술 이름을 섞으면 RAG 가 착각하기 쉬운 좋은 문항이 됩니다.

---
## 7단계. 최종 테스트셋 저장

**검수 통과 문항 + 문서에 없는 질문**을 합쳐 저장합니다.
이 파일은 앞으로 설정을 비교할 때 **바꾸지 않고 계속 쓰는 기준 테스트셋**입니다. (바꾸면 이전 점수와 비교할 수 없음)

In [ ]:
final = pd.concat(
    [human_pass[["user_input", "reference", "reference_contexts", "synthesizer_name"]], negatives],
    ignore_index=True,
)

FINAL_PATH = "../data/ragas_testset_validated.csv"
final.to_csv(FINAL_PATH, index=False)

print(f"최종 {len(final)}문항 저장: {FINAL_PATH}")
print(final["synthesizer_name"].value_counts())

---
## 정리

| 단계 | 걸러내는 문제 | 기준값 (조정 가능) |
|---|---|---|
| 1. 언어 | 영어·혼합 질문 | `KO_MIN = 0.5` |
| 2. 중복 | 같은 질문 반복 → 점수 편중 | `DUP_MIN = 0.7` (임베딩 모델마다 다름) |
| 3. 베끼기 | 너무 쉬운 질문 → 검색 점수 부풀림 | `COPY_MAX = 0.85` |
| 4. 정답 검증 | 틀린 정답 → 맞는 답을 틀렸다고 채점 | `REF_MIN = 0.8`, 채점 모델은 생성 모델과 다르게 |
| 5. 사람 검수 | 부자연스러움, 정답 누락 | 불합격률 20% 초과 시 재생성 |
| 6. 문서 밖 질문 | 환각 여부 확인 불가 | 직접 3~5개 이상 추가 |

**다음 단계**: `ragas_testset_validated.csv` 를 ch05_02 의 평가 코드에 넣어, 검증 전 테스트셋과 **점수가 어떻게 달라지는지** 비교해 보세요.